In [1]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
print(p)
q = df[:, 6]


misurato = ~np.isnan(df[:, 2])     
sigma_fuoco = 3.0                 
err_p = df[:, 8]
err_q = np.where(misurato, df[:, 7], np.sqrt(df[:, 7]**2 + sigma_fuoco**2)) 
# err a ymax =0



#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

[ 232.5  233.5  234.5  235.5  236.5  237.5  238.5  239.5  240.5  241.5
  242.5  243.5  244.5  245.5  246.5  247.5  248.5  249.5  250.5  251.5
  252.5  253.5  254.5  255.5  256.5  257.5  258.5  259.5  260.5  261.5
  262.5  263.5  264.5  265.5  267.5  269.5  271.5  273.5  275.5  277.5
  280.5  281.5  282.5  283.5  284.5  285.5  286.5  287.5  288.5  289.5
  290.5  291.5  292.5  293.5  294.5  295.5  297.5  299.5  301.5  306.5
  311.5  316.5  321.5  326.5  331.5  336.5  341.5  346.5  351.5  356.5
  361.5  366.5  371.5  376.5  381.5  386.5  391.5  396.5  401.5  406.5
  411.5  416.5  421.5  426.5  431.5  441.5  451.5  461.5  471.5  481.5
  491.5  501.5  511.5  521.5  531.5  541.5  551.5  561.5  571.5  581.5
  601.5  621.5  641.5  661.5  681.5  731.5  781.5  831.5  881.5  981.5
 1081.5 1181.5 1281.5 1381.5 1481.5 1631.5]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      147.984
NDf                       = Fit status: 0
Parametri stimati:
A 

Info in <TCanvas::Print>: png file fit_root.png has been created


In [2]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [3]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff


#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Fit status: 0
Parametri stimati:
const = 17.05023210147125 ± 2.8974486001312667
#mu = 0.44383728467173905 ± 0.08936274632110482
#sigma = 0.7375501423729045 ± 0.11452300915330804
Chi2 = 20.033286956116697
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      20.0333
NDf                       =           14
Edm                       =  2.66857e-08
NCalls                    =           92
const                     =      17.0502   +/-   2.89745     
mu                        =     0.443837   +/-   0.0893627   
sigma                     =      0.73755   +/-   0.114523     	 (limited)


-2.9937368031866605
0
-1.0515701206423473
1
-2.5473187985770505
61
-4.1591542505830565
62
-1.5442128798061263
63
-0.9442245588843368
64
-1.3278313838666012
65
-1.0018463343322692
66
-0.9315243615655099
67
-1.4234400815760357
68
-1.79077989300588
69
-1.3487169642651315
70
-1.2927816172445443
72
-1.650498389654964
73
-2.138111969998198
74
-2.4122503881765813
75

Info in <TCanvas::Print>: png file hist.png has been created


In [4]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma è compatibile con 1


In [5]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[-37.77684993 -35.95448231   0.68392152 -23.91411906 -16.53358734
 -13.66814124  -9.45636046 -12.72229254  -5.71197946 -12.07169481
 -11.47467759   1.92138046 -11.61714203  -5.6255348   -9.91952784
   4.721875     6.06717961   5.26490053   2.21599094   3.9906214
   0.41205239   5.07512951   3.45048327   7.96250616   3.83198142
   2.57885674   0.40004678  -0.90341555   3.91711002   5.2099329
   2.65430317   7.68078724   4.80388761   4.50054775   4.60482925
   5.36434113   5.44964897   7.03895505   6.63169587   8.91926143
   5.71963604   7.96421827   7.05213322   2.04803396   0.28253493
   5.38470408   2.85458288   4.18796523   6.3493122    4.39463983
   4.80747605   7.08741171   3.29278094   5.82425282   4.78097687
   6.6152305    5.95864659   6.43607868   3.52246306   4.5571988
   5.23624534  -7.70398372 -12.56408618  -4.66039937  -2.84745273
  -4.00173846  -3.01771744  -2.80466613  -4.28416499  -5.38808103
  -4.05693468  -2.2385683   -3.88705311  -4.96178636  -6.42674098
  -7.24983877

Info in <TCanvas::Print>: png file fit_line.png has been created
